GPT-2 Model Configuration

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50527,
    "context_length": 256,
    "stride": 128,
    "embedding_dim": 768,
    "num_layers": 12,
    "num_heads": 12,
    "dropout": 0.1,
    "qkv_bias": False
}

Load Dataset

In [3]:

file_path = "../the-verdict.txt"

with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

Tokenization

In [4]:
import tiktoken
import torch

tokenizer = tiktoken.get_encoding("gpt2")

In [5]:
token_ids = tokenizer.encode(text_data)

In [6]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

In [7]:
len(token_ids)

5145

Implementing Dataloader

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_len, stride):

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_len, stride):
            input_chunk = token_ids[i:i+max_len]
            target_chunk = token_ids[i+1:i+max_len+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    
def create_dataloader(text, tokenizer, batch_size, context_len, stride, shuffle, drop_last, num_workers):

    dataset = GPTDataset(text, tokenizer, context_len, stride)

    dataloader = DataLoader(

        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers    
    )

    return dataloader

In [11]:
train_ratio = 0.8

train_data = text_data[:int(train_ratio*len(text_data))]
test_data = text_data[int(train_ratio*len(text_data)):]

torch.manual_seed(123)

train_dataloader = create_dataloader(

    text=train_data,
    tokenizer=tokenizer,
    batch_size=2,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=True,
    drop_last=True,
    num_workers=0

)

test_dataloader = create_dataloader(
    
    text=test_data,
    tokenizer=tokenizer,
    batch_size=2,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=False,
    drop_last=False,
    num_workers=0

)

In [12]:
for x, y in train_dataloader:
    print(x.shape, y.shape)

for x, y in test_dataloader:
    print(x.shape, y.shape)


torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([1, 256]) torch.Size([1, 256])


Multi-Head Attention

In [13]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, num_heads, context_len, dropout, qkv_bias):
        super().__init__()

        assert(d_out % num_heads ==0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = (d_out // num_heads)

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # print("Keys:")
        # print(keys)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # print(queries)
        
        # print(values)

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_score = queries @ keys.transpose(2, 3)
        # print(attn_score)
        
        masked_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_score.masked_fill_(masked_bool, -torch.inf)
        # print(attn_score)

        attn_weights = torch.softmax(attn_score / (keys.shape[-1] ** 0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)
        # print(attn_weights)

        context_vector = attn_weights @ values
        context_vector = context_vector.transpose(1, 2)

        context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        # print(context_vector)

        return context_vector
        


Layer Normalization, GELU Activation Function and Feed Forward Neural Network

In [15]:
class LayerNormaliztion(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        x_norm = (x - mean)/torch.sqrt(var + self.eps)
        # print(x_norm)

        return self.scale * x_norm + self.shift


class GELUActivation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))
    
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["embedding_dim"], 4*cfg["embedding_dim"]),
            GELUActivation(),
            nn.Linear(4*cfg["embedding_dim"], cfg["embedding_dim"])

        )
        
    def forward(self, x):
        return self.layers(x)

Transformer Block

In [16]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg["embedding_dim"],
            d_out=cfg["embedding_dim"],
            num_heads=cfg["num_heads"],
            context_len=cfg["context_length"],
            dropout=cfg["dropout"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.norm1 = LayerNormaliztion(cfg["embedding_dim"])
        self.norm2 = LayerNormaliztion(cfg["embedding_dim"])
        self.ff = FeedForward(cfg)
        self.dropout = nn.Dropout(cfg["dropout"])
    
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut
        # print(x)

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

GPT Model Architecture

In [17]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout"])

        self.transformers = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )

        self.final_norm = LayerNormaliztion(cfg["embedding_dim"])
        self.final_output = nn.Linear(cfg["embedding_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_emb + pos_emb
        x = self.dropout(x)
        # print("Token Embeddings")
        # print(tok_emb)

        x = self.transformers(x)
        # print(x)

        x = self.final_norm(x)
        logits = self.final_output(x)

        return logits
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

Loss Function

In [18]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    # print(logits)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    # print(loss)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [19]:
def evaluate_model(model, train_loader, test_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(test_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

Generate Text Function

In [20]:
def generate_text(model, idx, max_new_tokens, context_len):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -context_len:]

        with torch.no_grad():
            logits = model(idx_cond)
            logits = logits[:, -1, :]

            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.argmax(probs, dim=-1, keepdim=True)

            idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [21]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_len = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text(
            model=model, idx=encoded,
            max_new_tokens=50, context_len=context_len
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

Training Loop for LLM

In [22]:
def train_model(model, train_loader, test_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, test_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        print(f"Epoch: {epoch + 1}")
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen = tokens_seen + input_batch.numel()
            global_step += 1

            if(global_step % eval_freq ==0):
                train_loss, test_loss = evaluate_model(model, train_loader, test_loader, device, eval_iter)

                train_losses.append(train_loss)
                test_losses.append(test_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Test loss {test_loss:.3f}")
                
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )
        
    
    return train_losses, test_losses, track_tokens_seen         


In [23]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [24]:
print(torch.cuda.memory_allocated()/1024**3, "GB allocated")
print(torch.cuda.memory_reserved()/1024**3, "GB reserved")

0.0 GB allocated
0.0 GB reserved


In [25]:
import time
start_time = time.time()

gpu = torch.device('cuda')
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)



model.to(gpu)
optimizer = torch.optim.AdamW(model.parameters(), lr = 0.0004, weight_decay=0.1)

num_epochs = 10

train_losses, test_losses, tokens_seen = train_model(
    model, train_dataloader, test_dataloader, optimizer, gpu, num_epochs, eval_freq=5, eval_iter=5, 
    start_context="Every Effort Moves You", tokenizer=tokenizer
)

end_time = time.time()

time_taken = (end_time - start_time)/60

print(f"Time Taken for Training in minutes: {time_taken} Minutes")

Epoch: 1
Ep 1 (Step 000000): Train loss 9.783, Test loss 9.979
Ep 1 (Step 000005): Train loss 8.053, Test loss 8.449
Ep 1 (Step 000010): Train loss 6.632, Test loss 7.286
Every Effort Moves You.                                                 
Epoch: 2
Ep 2 (Step 000015): Train loss 6.060, Test loss 6.898
Ep 2 (Step 000020): Train loss 11.143, Test loss 12.370
Ep 2 (Step 000025): Train loss 5.623, Test loss 6.843
Every Effort Moves You                                                  
Epoch: 3
Ep 3 (Step 000030): Train loss 5.506, Test loss 6.926
Ep 3 (Step 000035): Train loss 5.200, Test loss 6.724
Ep 3 (Step 000040): Train loss 5.017, Test loss 6.738
Every Effort Moves You of the picture of the picture                                            
Epoch: 4
Ep 4 (Step 000045): Train loss 4.591, Test loss 6.636
Ep 4 (Step 000050): Train loss 4.404, Test loss 6.663
Ep 4 (Step 000055): Train loss 3.476, Test loss 6.544
Every Effort Moves You of the picture--and a little a little that, and 

In [28]:
import tensorflow as tf
import tqdm